In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/bank-additional-full.csv", sep=";")
print(df.shape)
print(df["y"].value_counts())
print(df["campaign"].sum())

(41188, 21)
y
no     36548
yes     4640
Name: count, dtype: int64
105754


In [4]:
# the decomposition
n_customers = len(df)
n_subs = (df["y"] == "yes").sum()
total_contacts = df["campaign"].sum()

print("Customers contacted per subscriber:", round(n_customers / n_subs, 2))
print("Contacts per customer:", round(total_contacts / n_customers, 2))
print("Contacts per subscriber:", round(total_contacts / n_subs, 2))

Customers contacted per subscriber: 8.88
Contacts per customer: 2.57
Contacts per subscriber: 22.79


In [8]:
# share of contacts vs share of subscribers, by job
summary = df.groupby("job").agg(
    contacts=("campaign", "sum"),
    subscribers=("y", lambda s: (s == "yes").sum()),
)
summary["contact_share_%"] = (summary["contacts"] / total_contacts * 100).round(1)
summary["subscriber_share_%"] = (summary["subscribers"] / n_subs * 100).round(1)
summary["contacts_per_subscriber"] = (summary["contacts"] / summary["subscribers"]).round(1)
summary["gap"] = (summary["contact_share_%"] - summary["subscriber_share_%"]).round(1)
summary.sort_values("gap", ascending=False)

,contacts,subscribers,contact_share_%,subscriber_share_%,contacts_per_subscriber,gap
job,,,,,,
blue-collar,23676,638,22.4,13.8,37.1,8.6
services,10271,323,9.7,7.0,31.8,2.7
entrepreneur,3692,124,3.5,2.7,29.8,0.8
technician,17379,730,16.4,15.7,23.8,0.7
self-employed,3781,149,3.6,3.2,25.4,0.4
housemaid,2798,106,2.6,2.3,26.4,0.3
unknown,874,37,0.8,0.8,23.6,0.0
management,7240,328,6.8,7.1,22.1,-0.3
unemployed,2600,144,2.5,3.1,18.1,-0.6


In [9]:
# to find repeated contact bands
df["contact_band"] = pd.cut(df["campaign"], bins=[0, 2, 4, 100], labels=["1-2", "3-4", "5+"])

bands = df.groupby("contact_band", observed=True).agg(
    customers=("campaign", "size"),
    contacts=("campaign", "sum"),
    subscribers=("y", lambda s: (s == "yes").sum()),
)
bands["contact_share_%"] = (bands["contacts"] / total_contacts * 100).round(1)
bands["subscriber_share_%"] = (bands["subscribers"] / n_subs * 100).round(1)
bands["subscription_rate_%"] = (bands["subscribers"] / bands["customers"] * 100).round(1)
bands


,customers,contacts,subscribers,contact_share_%,subscriber_share_%,subscription_rate_%
contact_band,,,,,,
1-2,28212,38782,3511,36.7,75.7,12.4
3-4,7992,26627,823,25.2,17.7,10.3
5+,4984,40345,306,38.1,6.6,6.1


In [10]:
pd.DataFrame({
    "type": df.dtypes.astype(str),
    "unique_values": df.nunique(),
    "unknown_count": (df == "unknown").sum(),
})

,type,unique_values,unknown_count
age,int64,78,0
job,str,12,330
marital,str,4,80
education,str,8,1731
default,str,3,8597
housing,str,3,990
loan,str,3,990
contact,str,2,0
month,str,10,0
day_of_week,str,5,0


In [11]:
df["pdays"].eq(999).mean()

np.float64(0.9632174419733903)